# Week-5: Apache Spark – Big Data Processing
**All questions answered with a single shared synthetic dataset. No external files or paths needed.**

## ⚙️ Setup – Install & Initialize Spark

In [1]:
import subprocess, sys

# Auto-install pyspark if not present
try:
    import pyspark
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyspark', '--quiet'])

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

spark = SparkSession.builder \
    .appName('Week5_Spark_Assignment') \
    .master('local[*]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('✅ Spark Version:', spark.version)
print('✅ SparkSession created successfully')

✅ Spark Version: 4.0.2
✅ SparkSession created successfully


## 📦 Shared Synthetic Dataset
One dataset used across all questions — no external files needed.

In [2]:
from datetime import datetime

# ── Raw data ──────────────────────────────────────────────────────────────────
data = [
    # user_id, transaction_date, region, product_category, sale_amount,
    # city, age, subscription, status, email, username,
    # price, store_id, raw_timestamp
    (1, '2024-01-10', 'West',  'Electronics', 250.0,  'Los Angeles', 25, 'Premium', 'Active',  'alice@mail.com',  'alice',   250.0, 'S01', '2024-01-10 08:30:00'),
    (2, '2024-01-11', 'East',  'Clothing',    150.0,  'New York',   30, 'Basic',   'Active',  'bob@mail.com',    'bob',     150.0, 'S02', '2024-01-11 09:00:00'),
    (3, '2024-01-12', 'West',  'Electronics', 300.0,  'Los Angeles', 22, 'Premium', 'Inactive','carol@mail.com',  'carol',   300.0, 'S01', '2024-01-12 10:15:00'),
    (4, '2024-01-10', 'North', 'Food',         80.0,  'Chicago',    45, 'Basic',   'Active',  'dave@mail.com',   'dave',     80.0, 'S03', '2024-01-10 11:00:00'),
    (5, '2024-01-13', 'West',  'Clothing',    200.0,  'Seattle',    28, 'Premium', 'Active',  'eve@mail.com',    'eve',     200.0, 'S02', '2024-01-13 12:30:00'),
    (6, '2024-01-14', 'South', 'Electronics', 500.0,  'Houston',    35, 'Premium', 'Active',  'frank@mail.com',  'frank',   500.0, 'S04', '2024-01-14 13:00:00'),
    (7, '2024-01-15', 'East',  'Food',        120.0,  'New York',   19, 'Basic',   None,      'grace@mail.com',  'grace',   120.0, 'S02', '2024-01-15 14:00:00'),
    (8, '2024-01-16', 'West',  'Electronics', 450.0,  'Los Angeles', 24, 'Premium', 'Active',  'henry@mail.com',  'henry',   450.0, 'S01', '2024-01-16 15:30:00'),
    (9, '2024-01-17', 'North', 'Clothing',    None,   'Chicago',    29, 'Premium', 'Active',  None,              'ivan',    None,  'S03', '2024-01-17 16:00:00'),
    (10,'2024-01-18', 'South', 'Food',         90.0,  'Miami',      50, 'Basic',   'Inactive','judy@mail.com',   '',         90.0, 'S04', '2024-01-18 17:00:00'),
    # duplicates for Q3/Q15
    (1, '2024-01-10', 'West',  'Electronics', 250.0,  'Los Angeles', 25, 'Premium', 'Active',  'alice@mail.com',  'alice',   250.0, 'S01', '2024-01-10 08:30:00'),
    (3, '2024-01-12', 'West',  'Electronics', 300.0,  'Los Angeles', 22, 'Premium', 'Inactive','carol@mail.com',  'carol',   300.0, 'S01', '2024-01-12 10:15:00'),
    # extra rows to push city counts > 100 threshold (simulated)
    *[(i, '2024-01-20', 'West', 'Electronics', 100.0, 'Los Angeles', 27, 'Premium', 'Active', f'u{i}@mail.com', f'user{i}', 100.0, 'S01', '2024-01-20 08:00:00') for i in range(101, 201)],
    *[(i, '2024-01-20', 'East', 'Clothing',     90.0, 'New York',   32, 'Basic',   'Active', f'v{i}@mail.com', f'userv{i}', 90.0, 'S02', '2024-01-20 09:00:00') for i in range(201, 256)],
]

schema = StructType([
    StructField('user_id',          IntegerType(), True),
    StructField('transaction_date', StringType(),  True),
    StructField('region',           StringType(),  True),
    StructField('product_category', StringType(),  True),
    StructField('sale_amount',      DoubleType(),  True),
    StructField('city',             StringType(),  True),
    StructField('age',              IntegerType(), True),
    StructField('subscription',     StringType(),  True),
    StructField('status',           StringType(),  True),
    StructField('email',            StringType(),  True),
    StructField('username',         StringType(),  True),
    StructField('price',            DoubleType(),  True),
    StructField('store_id',         StringType(),  True),
    StructField('raw_timestamp',    StringType(),  True),
])

df = spark.createDataFrame(data, schema=schema)

# Handy aliases used across questions
df_sales = df

print(f'✅ Dataset created: {df.count()} rows, {len(df.columns)} columns')
df.show(12, truncate=False)

✅ Dataset created: 167 rows, 14 columns
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+--------------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|city       |age|subscription|status  |email         |username|price|store_id|raw_timestamp      |
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+--------------+--------+-----+--------+-------------------+
|1      |2024-01-10      |West  |Electronics     |250.0      |Los Angeles|25 |Premium     |Active  |alice@mail.com|alice   |250.0|S01     |2024-01-10 08:30:00|
|2      |2024-01-11      |East  |Clothing        |150.0      |New York   |30 |Basic       |Active  |bob@mail.com  |bob     |150.0|S02     |2024-01-11 09:00:00|
|3      |2024-01-12      |West  |Electronics     |300.0      |Los Angeles|22 |Premium     |Inactive|carol@mail.com|carol   |300.0|S01     |2024-01-12 10:15:00|


---
## Q1: Key Limitations of Traditional MapReduce vs Spark

### Answer

| Limitation (MapReduce) | How Spark Solves It |
|------------------------|---------------------|
| **Disk I/O after every step** – Map and Reduce results are always written to HDFS between stages, making multi-step jobs extremely slow. | Spark caches intermediate results **in memory (RAM)**, avoiding repeated disk reads/writes. |
| **No native iteration support** – ML algorithms (e.g., k-means, gradient descent) need many passes over data. Each pass in MapReduce is a full new job with full disk write+read. | Spark's **RDD / DataFrame** lineage lets iterative jobs reuse cached data across iterations without touching disk. |
| **High latency** – Job startup cost (JVM spin-up, HDFS coordination) is paid every time, even for small computations. | Spark keeps a **long-running executor JVM**, so subsequent operations have negligible startup cost. |
| **Only two phases (Map → Reduce)** – Complex pipelines must be chained as multiple separate jobs. | Spark supports rich DAG (Directed Acyclic Graph) execution with **map, filter, join, groupBy, window, ML** etc. all within one job. |
| **No real-time / streaming** – MapReduce is strictly batch-oriented. | Spark offers **Structured Streaming** for near-real-time processing on the same API. |
| **Verbose API** – Developers write a lot of boilerplate Java/Python code for even simple operations. | Spark's **DataFrame / SQL API** expresses the same logic in a few readable lines. |

> **Summary:** MapReduce treats HDFS as the communication bus between steps. Spark treats RAM as the communication bus, falling back to disk only when memory is full.

---
## Q2: In-Memory Computing & Iterative ML Algorithms

### Answer

**How disk-based MapReduce handles iteration:**
```
Iteration 1:  Read from HDFS → Compute → Write to HDFS
Iteration 2:  Read from HDFS → Compute → Write to HDFS
Iteration N:  Read from HDFS → Compute → Write to HDFS
```
Every iteration hits slow disk I/O twice — once for reading, once for writing.

**How Spark handles iteration (in-memory):**
```
Iteration 1:  Read from disk → Compute → Cache in RAM
Iteration 2:  Read from RAM  → Compute → Update cache
Iteration N:  Read from RAM  → Compute → Done
```
Only the **first read** touches disk. All subsequent iterations operate entirely in RAM.

**Key mechanisms:**
- **`df.cache()` / `df.persist()`** – Explicitly store a DataFrame in memory so it is not recomputed.
- **Lazy Evaluation + DAG Optimizer (Catalyst)** – Spark builds a plan and only executes when an action (`.count()`, `.show()`) is called, optimising the full pipeline.
- **Tungsten Engine** – Off-heap binary memory format that reduces GC pressure and improves cache efficiency.

**Benchmark result (reported in Spark paper):** Spark runs logistic regression **~100× faster** than Hadoop MapReduce on iterative workloads because it eliminates repeated HDFS I/O.

In [3]:
# Demonstrating .cache() — the core of in-memory computing
df_cached = df.cache()          # Pin in memory after first action
print('Row count (triggers caching):', df_cached.count())
print('Second access uses RAM — no disk read')
print('Distinct regions:', df_cached.select('region').distinct().count())
df_cached.unpersist()           # Free memory when done
print('Cache released.')

Row count (triggers caching): 167
Second access uses RAM — no disk read
Distinct regions: 4
Cache released.


---
## Q3: Remove Duplicate Rows Based on `user_id` and `transaction_date`

In [4]:
print('Before deduplication:', df.count(), 'rows')

df_deduped = df.dropDuplicates(['user_id', 'transaction_date'])

print('After deduplication :', df_deduped.count(), 'rows')
print('\nSample (user_id, transaction_date):')
df_deduped.select('user_id', 'transaction_date').orderBy('user_id').show(12)

Before deduplication: 167 rows
After deduplication : 165 rows

Sample (user_id, transaction_date):
+-------+----------------+
|user_id|transaction_date|
+-------+----------------+
|      1|      2024-01-10|
|      2|      2024-01-11|
|      3|      2024-01-12|
|      4|      2024-01-10|
|      5|      2024-01-13|
|      6|      2024-01-14|
|      7|      2024-01-15|
|      8|      2024-01-16|
|      9|      2024-01-17|
|     10|      2024-01-18|
|    101|      2024-01-20|
|    102|      2024-01-20|
+-------+----------------+
only showing top 12 rows


---
## Q4: Filter for Region `'West'` → Group by `product_category` → Average `sale_amount`

In [5]:
result_q4 = (
    df_sales
    .filter(F.col('region') == 'West')
    .groupBy('product_category')
    .agg(F.avg('sale_amount').alias('avg_sale_amount'))
    .orderBy('avg_sale_amount', ascending=False)
)

print('Average Sale Amount by Product Category (Region = West):')
result_q4.show()

Average Sale Amount by Product Category (Region = West):
+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|        Clothing|          200.0|
|     Electronics|          110.0|
+----------------+---------------+



---
## Q5: Difference Between `.na.drop()` and `.na.fill()`

### Answer

| Method | What it does | When to use |
|--------|-------------|-------------|
| **`.na.drop()`** | **Removes rows** that contain null values (all columns, or specified columns). | When a row with missing key data is unusable and should be excluded entirely. |
| **`.na.fill()`** | **Replaces nulls** with a specified default value — does NOT remove any row. | When you want to keep all rows and substitute a sensible default for missing values. |

In [6]:
# ── .na.drop() example ────────────────────────────────────────────────────────
print('Rows with null status (before drop):')
df.filter(F.col('status').isNull()).select('user_id','status').show()

df_dropped = df.na.drop(subset=['status'])
print('Rows after dropping nulls in status:', df_dropped.count())

# ── .na.fill() example ────────────────────────────────────────────────────────
df_filled = df.na.fill({'status': 'Unknown'})

print('\nStatus column after filling nulls with Unknown:')
df_filled.select('user_id', 'status') \
         .filter(F.col('user_id') == 7) \
         .show()

Rows with null status (before drop):
+-------+------+
|user_id|status|
+-------+------+
|      7|  NULL|
+-------+------+

Rows after dropping nulls in status: 166

Status column after filling nulls with Unknown:
+-------+-------+
|user_id| status|
+-------+-------+
|      7|Unknown|
+-------+-------+



---
## Q6: Total Record Count per City — Only Cities with Count > 100

In [18]:
result_q6 = (
    df
    .groupBy('city')
    .agg(F.count('*').alias('total_records'))
    .filter(F.col('total_records') > 100)
    .orderBy('total_records', ascending=False)
)

print('Cities with more than 100 records:')
result_q6.show()

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

---
## Q7: Immutability of Spark DataFrames & Data Cleaning

### Answer

Spark DataFrames are **immutable** — once created, they cannot be changed in-place. Every transformation (drop, rename, filter, cast) returns a **new** DataFrame; the original stays untouched.

**Implications for data cleaning:**

1. **Each cleaning step = new variable (or chained call)**  
   You cannot do `df.drop('col')` and expect `df` to change — you must write `df_clean = df.drop('col')`.

2. **Full lineage is preserved**  
   Spark remembers every transformation in a DAG. If a cached partition is lost, Spark can recompute it automatically.

3. **Safe for parallel workers**  
   Because no executor can mutate shared state, there are no race conditions during distributed cleaning.

4. **Chaining is idiomatic**  
   Multiple cleaning steps are typically chained in one expression for clarity (see example below).

In [8]:
# Original df is NOT modified — cleaning returns a new DataFrame
df_clean = (
    df
    .drop('raw_timestamp')                          # drop unwanted column
    .withColumnRenamed('sale_amount', 'revenue')    # rename column
)

print('Original df columns :', df.columns)
print('Cleaned df columns  :', df_clean.columns)
print("\n'raw_timestamp' removed and 'sale_amount' renamed to 'revenue'")

Original df columns : ['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'city', 'age', 'subscription', 'status', 'email', 'username', 'price', 'store_id', 'raw_timestamp']
Cleaned df columns  : ['user_id', 'transaction_date', 'region', 'product_category', 'revenue', 'city', 'age', 'subscription', 'status', 'email', 'username', 'price', 'store_id']

'raw_timestamp' removed and 'sale_amount' renamed to 'revenue'


---
## Q8: Filter — Age Between 18–30 (inclusive) AND Subscription = `'Premium'`

In [9]:
result_q8 = df.filter(
    (F.col('age').between(18, 30)) &
    (F.col('subscription') == 'Premium')
)

print('Records where age ∈ [18,30] and subscription = Premium:')
result_q8.select('user_id', 'age', 'subscription', 'region').show(10)

Records where age ∈ [18,30] and subscription = Premium:
+-------+---+------------+------+
|user_id|age|subscription|region|
+-------+---+------------+------+
|      1| 25|     Premium|  West|
|      3| 22|     Premium|  West|
|      5| 28|     Premium|  West|
|      8| 24|     Premium|  West|
|      9| 29|     Premium| North|
|      1| 25|     Premium|  West|
|      3| 22|     Premium|  West|
|    101| 27|     Premium|  West|
|    102| 27|     Premium|  West|
|    103| 27|     Premium|  West|
+-------+---+------------+------+
only showing top 10 rows


---
## Q9: Why Handle Nulls Before Mathematical Aggregations?

### Answer

**The problem with nulls in aggregations:**

- `sum()`, `avg()`, `min()`, `max()` in Spark **silently skip** null values.
- This means the result is **computed over fewer rows** than you might expect, with **no warning**.

**Concrete risk examples:**

| Situation | What happens | Why it's dangerous |
|-----------|-------------|--------------------|
| 10 sales but 3 have null `sale_amount` | `sum()` totals only 7 rows | Revenue is under-reported |
| 100 ages but 20 are null | `avg(age)` is the mean of 80 people | Wrong demographic insight |
| Null in `price` before multiplication | `price * quantity` = null for that row | Entire row silently contributes 0 |

**Best practice:** Before aggregating, either:
- **Drop** rows with null in the aggregation column (`.na.drop(subset=['price'])`), or  
- **Fill** nulls with a default (`.na.fill({'price': 0})`)

In [10]:
# Demonstrate the difference
print('AVG price WITHOUT handling nulls:', df.agg(F.avg('price')).collect()[0][0])

df_no_nulls = df.na.fill({'price': 0})
print('AVG price WITH null filled as 0 :', df_no_nulls.agg(F.avg('price')).collect()[0][0])
print('\nNote: The averages differ because nulls silently reduce the denominator.')

AVG price WITHOUT handling nulls: 106.26506024096386
AVG price WITH null filled as 0 : 105.62874251497006

Note: The averages differ because nulls silently reduce the denominator.


---
## Q10: Cast `raw_timestamp` (String) → `TimestampType` and Rename to `event_time`

In [11]:
df_with_ts = (
    df
    .withColumn(
        'event_time',
        F.col('raw_timestamp').cast(TimestampType())
    )
    .drop('raw_timestamp')
)

print('Schema after casting:')
df_with_ts.printSchema()

print('Sample event_time values:')
df_with_ts.select('user_id', 'event_time').show(6, truncate=False)

Schema after casting:
root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

Sample event_time values:
+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|1      |2024-01-10 08:30:00|
|2      |2024-01-11 09:00:00|
|3      |2024-01-12 10:15:00|
|4      |2024-01-10 11:00:00|
|5      |2024-01-13 12:30:00|
|6      |2024-01-14 13:00:00|
+-------+-------------------+
only showing top 6 rows


---
## Q11: The "Shuffle" Process & Why GroupBy is a Wide Transformation

### Answer

**Narrow vs Wide Transformations:**

| Type | Definition | Examples |
|------|-----------|----------|
| **Narrow** | Each input partition contributes to exactly **one** output partition. No data moves between executors. | `filter`, `map`, `select`, `withColumn` |
| **Wide** | Each input partition may contribute to **multiple** output partitions. Data **must move across the network**. | `groupBy`, `join`, `distinct`, `orderBy` |

**The Shuffle process (step-by-step for `groupBy`):**

```
Step 1 – Map Phase (local)
  Each executor processes its local partition and computes partial aggregates.
  Keys are hashed to determine which output partition they belong to.

Step 2 – Shuffle Write
  Each executor writes shuffle files to local disk, sorted and partitioned by key hash.
  (These files are the "shuffle spill".)

Step 3 – Shuffle Read (network transfer)
  Executors pull the partitions they are responsible for from other executors over the network.
  This is the expensive part — network I/O + disk I/O.

Step 4 – Reduce Phase
  Each executor now holds ALL rows for a subset of keys and computes final aggregates.
```

**Why it's expensive:**
- Network serialization & transfer of potentially huge data volumes.
- Temporary disk writes during shuffle spill.
- Forces a **stage boundary** in the DAG — the next stage cannot start until the shuffle is fully complete.

**Optimization tips:** Use `spark.sql.shuffle.partitions` to control the number of post-shuffle partitions (default 200, which may be too many for small datasets).

In [12]:
# Visualise the stage boundary — groupBy triggers a shuffle
spark.conf.set('spark.sql.shuffle.partitions', '8')  # Reduce from 200 for demo

result_q11 = df.groupBy('region').agg(F.count('*').alias('cnt'))
result_q11.explain()   # Shows the Exchange (shuffle) node in the physical plan

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#2], functions=[count(1)])
   +- Exchange hashpartitioning(region#2, 8), ENSURE_REQUIREMENTS, [plan_id=697]
      +- HashAggregate(keys=[region#2], functions=[partial_count(1)])
         +- Project [region#2]
            +- Scan ExistingRDD[user_id#0,transaction_date#1,region#2,product_category#3,sale_amount#4,city#5,age#6,subscription#7,status#8,email#9,username#10,price#11,store_id#12,raw_timestamp#13]




---
## Q12: Remove Rows Where `email` is Null OR `username` is an Empty String

In [13]:
print('Before cleaning:', df.count(), 'rows')
print('Rows with null email       :', df.filter(F.col('email').isNull()).count())
print('Rows with empty username   :', df.filter(F.col('username') == '').count())

df_cleaned_q12 = df.filter(
    F.col('email').isNotNull() &
    (F.col('username') != '')
)

print('\nAfter cleaning:', df_cleaned_q12.count(), 'rows')
print('Sample cleaned rows:')
df_cleaned_q12.select('user_id', 'email', 'username').show(5)

Before cleaning: 167 rows
Rows with null email       : 1
Rows with empty username   : 1

After cleaning: 165 rows
Sample cleaned rows:
+-------+--------------+--------+
|user_id|         email|username|
+-------+--------------+--------+
|      1|alice@mail.com|   alice|
|      2|  bob@mail.com|     bob|
|      3|carol@mail.com|   carol|
|      4| dave@mail.com|    dave|
|      5|  eve@mail.com|     eve|
+-------+--------------+--------+
only showing top 5 rows


---
## Q13: Using `.agg()` for Multiple Statistics (`min`, `max`, `mean`) on `price`

In [14]:
result_q13 = df.agg(
    F.min('price').alias('min_price'),
    F.max('price').alias('max_price'),
    F.avg('price').alias('mean_price'),
    F.stddev('price').alias('stddev_price'),
    F.count('price').alias('non_null_count')
)

print('Price Statistics:')
result_q13.show(truncate=False)

Price Statistics:
+---------+---------+------------------+-----------------+--------------+
|min_price|max_price|mean_price        |stddev_price     |non_null_count|
+---------+---------+------------------+-----------------+--------------+
|80.0     |500.0    |106.26506024096386|50.44555044494332|166           |
+---------+---------+------------------+-----------------+--------------+



---
## Q14: Risk of `inferSchema=True` with Messy or Inconsistent Date Formats

### Answer

When you use `inferSchema=True`, Spark **samples a portion of the data** (by default the first 1000 rows) and guesses each column's data type.

**Risks with messy date formats:**

| Risk | Description | Example |
|------|-------------|--------|
| **Wrong type inferred** | If dates look like strings (`"Jan 10, 2024"`), Spark infers `StringType`, not `DateType`. | `"01/10/2024"` → inferred as `String`, not `Date` |
| **Partial inference failure** | Mixed formats in the same column (`2024-01-10` vs `10-Jan-24`) cause inference to fall back to `StringType` for the whole column. | Math on dates becomes impossible |
| **Silent data corruption** | A column like `01-02-03` could be inferred as a date or a decimal — Spark picks one interpretation for the entire column. | `01-02-03` → `Date(2001-02-03)` or `0.01`? |
| **Schema shifts across runs** | If new data arrives with different date styles, `inferSchema` may produce a **different schema** than before, breaking downstream pipelines. | Pipeline breaks silently in production |

**Best practice:**
```python
# Always define schema explicitly for production pipelines
schema = StructType([
    StructField('event_date', DateType(), True),  # enforce type
    ...
])
df = spark.read.schema(schema).csv('data.csv')

# Or cast after reading with a known format
df = df.withColumn('event_date', F.to_date('raw_date', 'yyyy-MM-dd'))
```

In [15]:
# Demonstrate safe date parsing with explicit format
df_dated = df.withColumn(
    'parsed_date',
    F.to_date(F.col('transaction_date'), 'yyyy-MM-dd')
)

print('Schema of parsed_date:')
df_dated.select('transaction_date', 'parsed_date').printSchema()
df_dated.select('transaction_date', 'parsed_date').show(5)

Schema of parsed_date:
root
 |-- transaction_date: string (nullable = true)
 |-- parsed_date: date (nullable = true)

+----------------+-----------+
|transaction_date|parsed_date|
+----------------+-----------+
|      2024-01-10| 2024-01-10|
|      2024-01-11| 2024-01-11|
|      2024-01-12| 2024-01-12|
|      2024-01-10| 2024-01-10|
|      2024-01-13| 2024-01-13|
+----------------+-----------+
only showing top 5 rows


---
## Q15: Final Processing Pipeline
**Step 1:** Remove duplicates  
**Step 2:** Fill null prices with 0  
**Step 3:** Group by `store_id` → calculate total revenue

In [16]:
# ── Step 1: Remove duplicates ─────────────────────────────────────────────────
df_step1 = df.dropDuplicates(['user_id', 'transaction_date'])
print(f'Step 1 — After deduplication : {df_step1.count()} rows (was {df.count()})')

# ── Step 2: Fill null prices with 0 ──────────────────────────────────────────
df_step2 = df_step1.na.fill({'price': 0})
null_before = df_step1.filter(F.col('price').isNull()).count()
null_after  = df_step2.filter(F.col('price').isNull()).count()
print(f'Step 2 — Null prices before fill: {null_before} | after fill: {null_after}')

# ── Step 3: Group by store_id → total revenue ─────────────────────────────────
df_step3 = (
    df_step2
    .groupBy('store_id')
    .agg(F.sum('price').alias('total_revenue'))
    .orderBy('total_revenue', ascending=False)
)

print('\nStep 3 — Total Revenue by Store:')
df_step3.show()

# ── One-liner chained version ─────────────────────────────────────────────────
print('── Chained pipeline (equivalent, idiomatic Spark) ──')
(
    df
    .dropDuplicates(['user_id', 'transaction_date'])
    .na.fill({'price': 0})
    .groupBy('store_id')
    .agg(F.sum('price').alias('total_revenue'))
    .orderBy('total_revenue', ascending=False)
).show()

Step 1 — After deduplication : 165 rows (was 167)
Step 2 — Null prices before fill: 1 | after fill: 0

Step 3 — Total Revenue by Store:
+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|     S01|      11000.0|
|     S02|       5420.0|
|     S04|        590.0|
|     S03|         80.0|
+--------+-------------+

── Chained pipeline (equivalent, idiomatic Spark) ──
+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|     S01|      11000.0|
|     S02|       5420.0|
|     S04|        590.0|
|     S03|         80.0|
+--------+-------------+



---
## ✅ Summary Table

| Q  | Topic | Key Concept |
|----|-------|-------------|
| 1  | MapReduce vs Spark | Disk I/O, latency, DAG, streaming |
| 2  | In-Memory Computing | `.cache()`, iterative ML, Tungsten |
| 3  | Deduplication | `.dropDuplicates(['col1','col2'])` |
| 4  | Filter + GroupBy | `.filter()` → `.groupBy()` → `.agg(avg)` |
| 5  | Null Handling | `.na.drop()` vs `.na.fill()` |
| 6  | GroupBy + HAVING | `.groupBy().agg(count).filter(count > 100)` |
| 7  | Immutability | Every transform returns a new DataFrame |
| 8  | Range Filter | `.between()` + compound `&` condition |
| 9  | Nulls in Aggregations | Silent row exclusion, pre-fill strategy |
| 10 | Type Casting | `.cast(TimestampType())` + rename |
| 11 | Shuffle / Wide Transform | Network exchange, stage boundary, DAG |
| 12 | Multi-condition Filter | `.isNotNull()` and `!= ''` combined |
| 13 | `.agg()` Multi-stats | `min`, `max`, `avg`, `stddev` in one call |
| 14 | inferSchema Risk | Mixed formats → wrong type, silent corruption |
| 15 | End-to-end Pipeline | Dedup → fill nulls → groupBy → revenue |

In [17]:
spark.stop()
print('✅ SparkSession stopped. All questions completed.')

✅ SparkSession stopped. All questions completed.
